In [2]:
#check the session
print("lakehouse loaded susccesssfully")

StatementMeta(, 55b62151-9509-43bd-bad8-646e4b5743ac, 4, Finished, Available, Finished, False)

lakehouse loaded susccesssfully


In [1]:
products_df = spark.read.option("header", True).csv("Files/bronze/products.csv")

stores_df = spark.read.option("header", True).csv("Files/bronze/dark_stores.csv")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 3, Finished, Available, Finished, False)

In [2]:
display(products_df)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 21ba6f52-731f-414b-a7b1-7e5d180c4639)

In [3]:
display(stores_df)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7bdb4df4-578f-4f1a-99bd-b6c3523a7c67)

In [5]:
products_df.printSchema()
stores_df.printSchema()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 7, Finished, Available, Finished, False)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)

root
 |-- dark_store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- area: string (nullable = true)



In [8]:
#reading all the three files 
# Read Orders
orders_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("Files/bronze/orders_raw.csv")
)

# Read Products
products_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("Files/bronze/products.csv")
)

# Read Dark Stores
stores_df = (
    spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("Files/bronze/dark_stores.csv")
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 10, Finished, Available, Finished, False)

In [9]:
#print
display(orders_df)
display(products_df)
display(stores_df)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b7aca7f2-94b1-40ee-a41f-07cb24af017b)

SynapseWidget(Synapse.DataFrame, 349e82c0-c81a-4cf3-babf-0294901d3987)

SynapseWidget(Synapse.DataFrame, fd3af889-e425-4f2a-b066-124cd151d149)

In [10]:
#check the row count 
print("Orders :", orders_df.count())
print("Products :", products_df.count())
print("Stores :", stores_df.count())

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 12, Finished, Available, Finished, False)

Orders : 4120
Products : 25
Stores : 8


In [11]:
#creating the bronze delta tables 
orders_df.write.mode("overwrite").format("delta").saveAsTable("bronze_orders")

products_df.write.mode("overwrite").format("delta").saveAsTable("bronze_products")

stores_df.write.mode("overwrite").format("delta").saveAsTable("bronze_stores")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 13, Finished, Available, Finished, False)

In [12]:
spark.sql("SHOW TABLES").show()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 14, Finished, Available, Finished, False)

+--------------------+---------------+-----------+
|           namespace|      tableName|isTemporary|
+--------------------+---------------+-----------+
|blinkit_ai.blinki...|  bronze_orders|      false|
|blinkit_ai.blinki...|bronze_products|      false|
|blinkit_ai.blinki...|  bronze_stores|      false|
+--------------------+---------------+-----------+



In [13]:
#querying the table 
spark.sql("SELECT * FROM bronze_orders LIMIT 5").show()

spark.sql("SELECT * FROM bronze_products LIMIT 5").show()

spark.sql("SELECT * FROM bronze_stores LIMIT 5").show()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 15, Finished, Available, Finished, False)

+---------+-----------+-------------+----------+--------+---------------+----------------+-------------------+------------+--------------+
| order_id|customer_id|dark_store_id|product_id|quantity|order_timestamp|promised_minutes|delivered_timestamp|order_amount|payment_method|
+---------+-----------+-------------+----------+--------+---------------+----------------+-------------------+------------+--------------+
|ORD100088|   CUST0309|         DS02|       P18|       2| 6/2/2025 20:04|              15|     6/2/2025 20:10|         340|         Cash |
|ORD103464|   CUST0930|         DS06|       P15|       2|6/23/2025 19:05|              10|    6/23/2025 19:15|         280|          Card|
|ORD101652|   CUST0891|         DS07|       P16|       2|6/13/2025 19:44|              10|    6/13/2025 19:52|         112|          CASH|
|ORD101681|   CUST0972|         DS08|       P18|       1|6/28/2025 13:59|              10|    6/28/2025 14:08|         165|           upi|
|ORD100437|   CUST1048|    

# Silver Layer - Data Cleaning & Transformation

**Objective**

Transform raw Bronze data into clean, validated, and analytics-ready Silver tables.

In [27]:
#read the bronze tables
bronze_orders = spark.table("bronze_orders")
bronze_products = spark.table("bronze_products")
bronze_stores = spark.table("bronze_stores")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 29, Finished, Available, Finished, False)

In [28]:
bronze_orders.printSchema()
bronze_products.printSchema()
bronze_stores.printSchema()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 30, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- dark_store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- promised_minutes: integer (nullable = true)
 |-- delivered_timestamp: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- payment_method: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: integer (nullable = true)

root
 |-- dark_store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- area: string (nullable = true)



In [29]:
clean = bronze_orders

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 31, Finished, Available, Finished, False)

In [30]:
#removing the duplicate values 
clean = clean.dropDuplicates(["order_id"])

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 32, Finished, Available, Finished, False)

In [31]:
#remove the incomplete deliveries 
clean = clean.filter(
    F.col("delivered_timestamp").isNotNull()
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 33, Finished, Available, Finished, False)

In [32]:
#standardize the payements 
clean = clean.withColumn(
    "payment_method",
    F.upper(
        F.trim(
            F.col("payment_method")
        )
    )
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 34, Finished, Available, Finished, False)

In [33]:
clean.select("payment_method").distinct().show()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 35, Finished, Available, Finished, False)

+--------------+
|payment_method|
+--------------+
|          CARD|
|          CASH|
|        WALLET|
|           UPI|
+--------------+



In [34]:
#coverting the Date and time columns 
clean = (
    clean
    .withColumn(
        "order_ts",
        F.to_timestamp("order_timestamp")
    )
    .withColumn(
        "delivered_ts",
        F.to_timestamp("delivered_timestamp")
    )
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 36, Finished, Available, Finished, False)

In [35]:
clean.select(
    "order_timestamp",
    "order_ts",
    "delivered_timestamp",
    "delivered_ts"
).show(5, truncate=False)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 37, Finished, Available, Finished, False)

+---------------+--------+-------------------+------------+
|order_timestamp|order_ts|delivered_timestamp|delivered_ts|
+---------------+--------+-------------------+------------+
|6/19/2025 18:48|NULL    |6/19/2025 18:55    |NULL        |
|6/9/2025 20:34 |NULL    |6/9/2025 20:44     |NULL        |
|6/28/2025 20:29|NULL    |6/28/2025 20:40    |NULL        |
|6/6/2025 12:59 |NULL    |6/6/2025 13:05     |NULL        |
|6/29/2025 12:41|NULL    |6/29/2025 12:48    |NULL        |
+---------------+--------+-------------------+------------+
only showing top 5 rows



In [36]:
#time stamp is inconsist so 
clean = (
    clean
    .withColumn(
        "order_ts",
        F.to_timestamp(
            "order_timestamp",
            "M/d/yyyy HH:mm"
        )
    )
    .withColumn(
        "delivered_ts",
        F.to_timestamp(
            "delivered_timestamp",
            "M/d/yyyy HH:mm"
        )
    )
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 38, Finished, Available, Finished, False)

In [37]:
clean.select(
    "order_timestamp",
    "order_ts",
    "delivered_timestamp",
    "delivered_ts"
).show(5, truncate=False)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 39, Finished, Available, Finished, False)

+---------------+-------------------+-------------------+-------------------+
|order_timestamp|order_ts           |delivered_timestamp|delivered_ts       |
+---------------+-------------------+-------------------+-------------------+
|6/19/2025 18:48|2025-06-19 18:48:00|6/19/2025 18:55    |2025-06-19 18:55:00|
|6/9/2025 20:34 |2025-06-09 20:34:00|6/9/2025 20:44     |2025-06-09 20:44:00|
|6/28/2025 20:29|2025-06-28 20:29:00|6/28/2025 20:40    |2025-06-28 20:40:00|
|6/6/2025 12:59 |2025-06-06 12:59:00|6/6/2025 13:05     |2025-06-06 13:05:00|
|6/29/2025 12:41|2025-06-29 12:41:00|6/29/2025 12:48    |2025-06-29 12:48:00|
+---------------+-------------------+-------------------+-------------------+
only showing top 5 rows



In [38]:
#calculate the delivery time 
clean = clean.withColumn(
    "delivery_minutes",
    (
        F.unix_timestamp("delivered_ts") -
        F.unix_timestamp("order_ts")
    ) / 60
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 40, Finished, Available, Finished, False)

In [39]:
clean.select(
    "order_id",
    "promised_minutes",
    "delivery_minutes"
).show(10)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 41, Finished, Available, Finished, False)

+---------+----------------+----------------+
| order_id|promised_minutes|delivery_minutes|
+---------+----------------+----------------+
|ORD100001|              10|             7.0|
|ORD100002|              10|            10.0|
|ORD100003|              10|            11.0|
|ORD100004|              10|             6.0|
|ORD100005|              10|             7.0|
|ORD100006|              10|            10.0|
|ORD100007|              10|            21.0|
|ORD100008|              10|             6.0|
|ORD100009|              10|            13.0|
|ORD100010|              10|            10.0|
+---------+----------------+----------------+
only showing top 10 rows



we can see the late deliveries. we will add a late deliver flag 

In [40]:
clean = clean.withColumn(
    "is_late",
    (
        F.col("delivery_minutes") >
        F.col("promised_minutes")
    ).cast("int")
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 42, Finished, Available, Finished, False)

In [41]:
clean.select(
    "delivery_minutes",
    "promised_minutes",
    "is_late"
).show(10)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 43, Finished, Available, Finished, False)

+----------------+----------------+-------+
|delivery_minutes|promised_minutes|is_late|
+----------------+----------------+-------+
|             7.0|              10|      0|
|            10.0|              10|      0|
|            11.0|              10|      1|
|             6.0|              10|      0|
|             7.0|              10|      0|
|            10.0|              10|      0|
|            21.0|              10|      1|
|             6.0|              10|      0|
|            13.0|              10|      1|
|            10.0|              10|      0|
+----------------+----------------+-------+
only showing top 10 rows



In [48]:
clean = (
    clean
    .withColumn(
        "order_ts",
        F.to_timestamp("order_timestamp", "M/d/yyyy H:mm")
    )
    .withColumn(
        "delivered_ts",
        F.to_timestamp("delivered_timestamp", "M/d/yyyy H:mm")
    )
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 50, Finished, Available, Finished, False)

found the problem --
The format "M/d/yyyy H:mm" should parse both, but Microsoft Fabric's Spark sometimes throws parser exceptions because of its strict datetime parser.

In [51]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 53, Finished, Available, Finished, False)

In [52]:
clean = (
    clean
    .withColumn(
        "order_ts",
        F.to_timestamp(F.col("order_timestamp"), "M/d/yyyy H:mm")
    )
    .withColumn(
        "delivered_ts",
        F.to_timestamp(F.col("delivered_timestamp"), "M/d/yyyy H:mm")
    )
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 54, Finished, Available, Finished, False)

In [55]:
clean.select(
    "order_timestamp",
    "order_ts",
    "delivered_timestamp",
    "delivered_ts"
).show(10, truncate=False)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 57, Finished, Available, Finished, False)

+---------------+-------------------+-------------------+-------------------+
|order_timestamp|order_ts           |delivered_timestamp|delivered_ts       |
+---------------+-------------------+-------------------+-------------------+
|6/19/2025 18:48|2025-06-19 18:48:00|6/19/2025 18:55    |2025-06-19 18:55:00|
|6/9/2025 20:34 |2025-06-09 20:34:00|6/9/2025 20:44     |2025-06-09 20:44:00|
|6/28/2025 20:29|2025-06-28 20:29:00|6/28/2025 20:40    |2025-06-28 20:40:00|
|6/6/2025 12:59 |2025-06-06 12:59:00|6/6/2025 13:05     |2025-06-06 13:05:00|
|6/29/2025 12:41|2025-06-29 12:41:00|6/29/2025 12:48    |2025-06-29 12:48:00|
|6/13/2025 19:32|2025-06-13 19:32:00|6/13/2025 19:42    |2025-06-13 19:42:00|
|6/17/2025 18:43|2025-06-17 18:43:00|6/17/2025 19:04    |2025-06-17 19:04:00|
|6/29/2025 22:48|2025-06-29 22:48:00|6/29/2025 22:54    |2025-06-29 22:54:00|
|6/30/2025 20:15|2025-06-30 20:15:00|6/30/2025 20:28    |2025-06-30 20:28:00|
|6/6/2025 21:13 |2025-06-06 21:13:00|6/6/2025 21:23     |2025-06

In [56]:
clean = clean.withColumn(
    "order_hour",
    F.hour("order_ts")
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 58, Finished, Available, Finished, False)

In [57]:
#join the products and store 
silver_orders = (
    clean
    .join(bronze_products, on="product_id", how="left")
    .join(bronze_stores, on="dark_store_id", how="left")
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 59, Finished, Available, Finished, False)

In [58]:
print("Rows after join:", silver_orders.count())
display(silver_orders.limit(5))

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 60, Finished, Available, Finished, False)

Rows after join: 3833


SynapseWidget(Synapse.DataFrame, b92e156b-ff4c-4e8d-82a5-5e647f091bbe)

In [59]:
# select the business columns 
silver_orders = silver_orders.select(
    "order_id",
    "customer_id",
    "order_ts",
    "order_hour",
    "dark_store_id",
    "store_name",
    "city",
    "area",
    "product_id",
    "product_name",
    "category",
    "quantity",
    "price",
    "order_amount",
    "payment_method",
    "promised_minutes",
    "delivery_minutes",
    "is_late"
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 61, Finished, Available, Finished, False)

In [60]:
silver_orders.printSchema()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 62, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- order_hour: integer (nullable = true)
 |-- dark_store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- area: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- promised_minutes: integer (nullable = true)
 |-- delivery_minutes: double (nullable = true)
 |-- is_late: integer (nullable = true)



In [61]:
display(silver_orders.limit(10))

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 15ab8569-af0b-4272-a023-0c70db448c32)

In [62]:
silver_orders.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("silver_orders")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 64, Finished, Available, Finished, False)

In [63]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 65, Finished, Available, Finished, False)

+--------------------------------+---------------+-----------+
|namespace                       |tableName      |isTemporary|
+--------------------------------+---------------+-----------+
|blinkit_ai.blinkit_lakehouse.dbo|bronze_orders  |false      |
|blinkit_ai.blinkit_lakehouse.dbo|bronze_products|false      |
|blinkit_ai.blinkit_lakehouse.dbo|bronze_stores  |false      |
|blinkit_ai.blinkit_lakehouse.dbo|silver_orders  |false      |
+--------------------------------+---------------+-----------+



In [64]:
#validate the silver table 
print("Silver Orders:", silver_orders.count())

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 66, Finished, Available, Finished, False)

Silver Orders: 3833


In [65]:
#once again check for nulls
from pyspark.sql.functions import col, count, when

silver_orders.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_orders.columns
]).show()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 67, Finished, Available, Finished, False)

+--------+-----------+--------+----------+-------------+----------+----+----+----------+------------+--------+--------+-----+------------+--------------+----------------+----------------+-------+
|order_id|customer_id|order_ts|order_hour|dark_store_id|store_name|city|area|product_id|product_name|category|quantity|price|order_amount|payment_method|promised_minutes|delivery_minutes|is_late|
+--------+-----------+--------+----------+-------------+----------+----+----+----------+------------+--------+--------+-----+------------+--------------+----------------+----------------+-------+
|       0|          0|       0|         0|            0|         0|   0|   0|         0|           0|       0|       0|    0|          52|             0|               0|               0|      0|
+--------+-----------+--------+----------+-------------+----------+----+----+----------+------------+--------+--------+-----+------------+--------------+----------------+----------------+-------+



In [66]:
silver_orders.groupBy("payment_method").count().show()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 68, Finished, Available, Finished, False)

+--------------+-----+
|payment_method|count|
+--------------+-----+
|          CARD| 1160|
|          CASH|  767|
|        WALLET|  736|
|           UPI| 1170|
+--------------+-----+



In [123]:
silver_orders.groupBy("is_late").count().show()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 125, Finished, Available, Finished, False)

+-------+-----+
|is_late|count|
+-------+-----+
|      1| 1270|
|      0| 2563|
+-------+-----+



In [124]:
from pyspark.sql.functions import *

ml_df = spark.table("silver_orders")

display(ml_df)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 126, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10500ae4-3066-4ea6-9dad-c39ffdff876e)

In [125]:
ml_df = ml_df.drop(
    "order_id",
    "customer_id",
    "product_id"
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 127, Finished, Available, Finished, False)

In [126]:
ml_df = ml_df.withColumn(
    "order_amount",
    col("quantity") * col("price")
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 128, Finished, Available, Finished, False)

In [127]:
ml_df = ml_df.withColumn(
    "order_ts",
    to_timestamp("order_ts")
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 129, Finished, Available, Finished, False)

In [128]:
ml_df = ml_df.withColumn(
    "day_of_week",
    dayofweek("order_ts")
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 130, Finished, Available, Finished, False)

In [129]:
ml_df = ml_df.withColumn(
    "is_weekend",
    when(col("day_of_week").isin([1,7]),1).otherwise(0)
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 131, Finished, Available, Finished, False)

In [130]:
ml_df = ml_df.withColumn(
    "peak_hour",
    when(
        ((col("order_hour")>=11)&(col("order_hour")<=14))|
        ((col("order_hour")>=18)&(col("order_hour")<=22)),
        1
    ).otherwise(0)
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 132, Finished, Available, Finished, False)

In [131]:
ml_df = ml_df.dropna()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 133, Finished, Available, Finished, False)

In [132]:
train_df,test_df = ml_df.randomSplit([0.8,0.2],seed=42)

print(train_df.count())
print(test_df.count())

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 134, Finished, Available, Finished, False)

3121
712


In [133]:
from pyspark.ml import Pipeline

from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)

from pyspark.ml.classification import GBTClassifier

from pyspark.ml.evaluation import BinaryClassificationEvaluator

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 135, Finished, Available, Finished, False)

In [134]:
categorical_columns = [
    "payment_method",
    "category",
    "city",
    "area",
    "store_name",
    "product_name",
    "dark_store_id"
]

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 136, Finished, Available, Finished, False)

In [135]:
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c+"_idx",
        handleInvalid="keep"
    )
    for c in categorical_columns
]

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 137, Finished, Available, Finished, False)

In [136]:
encoder = OneHotEncoder(
    inputCols=[c+"_idx" for c in categorical_columns],
    outputCols=[c+"_vec" for c in categorical_columns]
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 138, Finished, Available, Finished, False)

In [137]:
numeric_columns = [
    "quantity",
    "price",
    "promised_minutes",
    "order_hour",
    "day_of_week",
    "is_weekend",
    "peak_hour"
]

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 139, Finished, Available, Finished, False)

In [138]:
assembler = VectorAssembler(
    inputCols=numeric_columns +
              [c+"_vec" for c in categorical_columns],
    outputCol="features",
    handleInvalid="keep"
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 140, Finished, Available, Finished, False)

In [139]:
gbt = GBTClassifier(
    labelCol="is_late",
    featuresCol="features",
    predictionCol="prediction",
    maxIter=150,
    maxDepth=8,
    maxBins=128,
    stepSize=0.1,
    seed=42
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 141, Finished, Available, Finished, False)

In [140]:
pipeline = Pipeline(
    stages=indexers + [encoder,assembler,gbt]
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 142, Finished, Available, Finished, False)

In [141]:
model = pipeline.fit(train_df)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 143, Finished, Available, Finished, False)

In [142]:
predictions = model.transform(test_df)

display(
    predictions.select(
        "is_late",
        "prediction",
        "probability"
    )
)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 144, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8a7a56d2-3722-4b46-88dd-642e2a0d721a)

In [143]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy = MulticlassClassificationEvaluator(
    labelCol="is_late",
    predictionCol="prediction",
    metricName="accuracy"
)

print("Accuracy:", accuracy.evaluate(predictions))

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 145, Finished, Available, Finished, False)

Accuracy: 0.6221910112359551


In [144]:
precision = MulticlassClassificationEvaluator(
    labelCol="is_late",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

print("Precision:", precision.evaluate(predictions))

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 146, Finished, Available, Finished, False)

Precision: 0.5953816920026437


In [145]:
recall = MulticlassClassificationEvaluator(
    labelCol="is_late",
    predictionCol="prediction",
    metricName="weightedRecall"
)

print("Recall:", recall.evaluate(predictions))

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 147, Finished, Available, Finished, False)

Recall: 0.622191011235955


In [146]:
f1 = MulticlassClassificationEvaluator(
    labelCol="is_late",
    predictionCol="prediction",
    metricName="f1"
)

print("F1:", f1.evaluate(predictions))

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 148, Finished, Available, Finished, False)

F1: 0.6048061555524133


In [147]:
predictions.groupBy(
    "is_late",
    "prediction"
).count().show()

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 149, Finished, Available, Finished, False)

+-------+----------+-----+
|is_late|prediction|count|
+-------+----------+-----+
|      1|       0.0|  164|
|      0|       0.0|  380|
|      1|       1.0|   63|
|      0|       1.0|  105|
+-------+----------+-----+



In [148]:
predictions.select(
    "order_ts",
    "store_name",
    "city",
    "product_name",
    "is_late",
    "prediction",
    "probability"
).write.mode("overwrite").saveAsTable("predictions_orders")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 150, Finished, Available, Finished, False)

### One important note

With the columns available in your dataset, do not expect 90%+ accuracy without introducing data leakage. Since delivery_minutes directly determines is_late, including it as a feature would artificially inflate the score and produce a model that cannot be used for real-world prediction.

The pipeline above is a valid predictive approach. If the accuracy is still around 65–75%, the limiting factor is likely the information available in the dataset rather than the Spark ML pipeline itself.

## Gold Layer

We'll create business-ready tables that Power BI can consume directly.

In [149]:
# 1. Store Performance 
from pyspark.sql.functions import *

gold_store = (
    ml_df.groupBy(
        "dark_store_id",
        "store_name",
        "city",
        "area"
    )
    .agg(
        count("*").alias("Total_Orders"),
        round(sum("order_amount"),2).alias("Revenue"),
        round(avg("delivery_minutes"),2).alias("Avg_Delivery_Time"),
        round(avg("promised_minutes"),2).alias("Avg_Promised_Time"),
        round(avg("quantity"),2).alias("Avg_Items"),
        sum("is_late").alias("Late_Orders")
    )
)

gold_store = gold_store.withColumn(
    "On_Time_Orders",
    col("Total_Orders")-col("Late_Orders")
)

gold_store = gold_store.withColumn(
    "Late_Percentage",
    round(
        col("Late_Orders")/col("Total_Orders")*100,
        2
    )
)

gold_store = gold_store.withColumn(
    "OnTime_Percentage",
    round(
        col("On_Time_Orders")/col("Total_Orders")*100,
        2
    )
)

display(gold_store)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 151, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7c4a5aec-8171-48e2-bc68-13fa23c2bd26)

In [150]:
gold_store.write.mode("overwrite").saveAsTable("gold_store_performance")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 152, Finished, Available, Finished, False)

In [151]:
gold_product = (
    ml_df.groupBy(
        "product_name",
        "category"
    )
    .agg(
        count("*").alias("Orders"),
        sum("quantity").alias("Total_Quantity"),
        round(sum("order_amount"),2).alias("Revenue"),
        round(avg("price"),2).alias("Average_Price")
    )
)

display(gold_product)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 153, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d723e997-a2f5-4ef2-b282-af52cfd5a535)

In [152]:
gold_product.write.mode("overwrite").saveAsTable("gold_product_performance")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 154, Finished, Available, Finished, False)

In [153]:
#hourly sales 
gold_hourly = (
    ml_df.groupBy("order_hour")
    .agg(
        count("*").alias("Orders"),
        round(sum("order_amount"),2).alias("Revenue"),
        round(avg("delivery_minutes"),2).alias("Avg_Delivery")
    )
    .orderBy("order_hour")
)

display(gold_hourly)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 155, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 45e29f74-0a23-47e8-b92b-2ab58b5dfa42)

In [154]:
gold_hourly.write.mode("overwrite").saveAsTable("gold_hourly_sales")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 156, Finished, Available, Finished, False)

In [155]:
gold_category = (
    ml_df.groupBy("category")
    .agg(
        count("*").alias("Orders"),
        round(sum("order_amount"),2).alias("Revenue"),
        round(avg("delivery_minutes"),2).alias("Avg_Delivery")
    )
)

display(gold_category)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 157, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a07d715d-c292-43c0-b0ae-b0ba698fb8a9)

In [156]:
gold_category.write.mode("overwrite").saveAsTable("gold_category_performance")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 158, Finished, Available, Finished, False)

In [157]:
#city performace 
gold_city = (
    ml_df.groupBy("city")
    .agg(
        count("*").alias("Orders"),
        round(sum("order_amount"),2).alias("Revenue"),
        round(avg("delivery_minutes"),2).alias("Avg_Delivery"),
        sum("is_late").alias("Late_Orders")
    )
)

display(gold_city)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 159, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7a6c3a58-f400-41f4-a671-dacd7f533e8a)

In [158]:
#ml predictions 
predictions.select(
    "order_ts",
    "store_name",
    "city",
    "category",
    "product_name",
    "quantity",
    "price",
    "order_amount",
    "is_late",
    "prediction",
    "probability"
).write.mode("overwrite").saveAsTable("gold_predictions")

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 160, Finished, Available, Finished, False)

In [159]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, f5a7f010-1442-4720-a1e4-826c5ac7ad18, 161, Finished, Available, Finished, False)

+--------------------------------+-------------------------+-----------+
|namespace                       |tableName                |isTemporary|
+--------------------------------+-------------------------+-----------+
|blinkit_ai.blinkit_lakehouse.dbo|bronze_orders            |false      |
|blinkit_ai.blinkit_lakehouse.dbo|bronze_products          |false      |
|blinkit_ai.blinkit_lakehouse.dbo|bronze_stores            |false      |
|blinkit_ai.blinkit_lakehouse.dbo|gold_category_performance|false      |
|blinkit_ai.blinkit_lakehouse.dbo|gold_hourly_sales        |false      |
|blinkit_ai.blinkit_lakehouse.dbo|gold_predictions         |false      |
|blinkit_ai.blinkit_lakehouse.dbo|gold_product_performance |false      |
|blinkit_ai.blinkit_lakehouse.dbo|gold_store_performance   |false      |
|blinkit_ai.blinkit_lakehouse.dbo|predictions_orders       |false      |
|blinkit_ai.blinkit_lakehouse.dbo|silver_orders            |false      |
+--------------------------------+-----------------